# HAIM-Guard: Threat-Aware Detection of AI-Generated and Hybrid Music

**Author:** Manus AI  
**Runtime:** Google Colab or a local Python 3.10+ notebook  
**Primary dataset:** [HAIM—Human–AI Music](https://huggingface.co/datasets/mippia/HAIM)  
**License reminder:** HAIM data are **CC BY-NC 4.0** and intended for non-commercial academic research.

This notebook builds a compact forensic triage system for music-industry cybersecurity. It focuses on a measurable control: detecting **fully synthetic music, unseen generators, post-processed synthetic music, and temporal human–AI mixtures**. It combines a supervised spectral detector, a real-only anomaly detector inspired by **MusicDET**, a shift-insensitive log-frequency fingerprint branch inspired by public Fourier-artifact research, calibrated thresholds, and a human-review abstention policy.

> **Important boundary:** A detector score is not proof of copyright infringement, impersonation, lack of consent, or unlawful conduct. Use it as an auditable triage signal with human review, provenance metadata, and an appeal path.

The notebook deliberately avoids reporting only random-split accuracy. It tests generator-disjoint generalization, hybrid production, codec and signal-processing laundering, calibration, false-positive control, and selective abstention.


## Why HAIM is the primary dataset

HAIM is the most representative public benchmark located for the stated security problem because it covers the full production chain: human music, fully AI music from multiple generators, AI mastering of human tracks, human mastering or mixing of AI tracks, AI vocal covers, AI variation/edit/repaint, and human–AI temporal mixtures.[1] The current public release contains **153,686 track records**, of which **67,000 include audio** (approximately 240–250 GB) and **86,686 are link-based**.[2] The paper's earlier 196,000 count and the current dataset-card count differ, so this notebook treats the current public release as authoritative for download planning.[1] [2]

| Mode | What it downloads | Recommended use |
| --- | --- | --- |
| **Quick Mode (default)** | A configurable sample from each official 100-row audio preview split | Colab demonstration, model debugging, threat analysis |
| **Research Mode (optional)** | Selected raw HAIM folders via `snapshot_download` | Larger experiments with adequate disk and compute |
| **Full corpus** | Approximately 245 GB | Dedicated storage; not recommended for ordinary Colab |

**Direct dataset links:** [HAIM download page](https://huggingface.co/datasets/mippia/HAIM), [official repository](https://github.com/Mippia/HAIM_dataset), and [paper](https://arxiv.org/abs/2606.01686).


In [ ]:
#@title 1. Install reproducible dependencies
%pip -q install "librosa>=0.10.2" "soundfile>=0.12" "scikit-learn>=1.4,<2" \
    "pandas>=2.0" "pyarrow>=15" "seaborn>=0.13" "joblib>=1.3" \
    "huggingface_hub>=0.25" "requests>=2.31" "tqdm>=4.66"

!which ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)


In [ ]:
#@title 2. Imports, seed, and experiment configuration
from __future__ import annotations

import io
import json
import math
import hashlib
import os
import random
import shutil
import subprocess
import tempfile
import time
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import soundfile as sf

from scipy.ndimage import median_filter
from scipy.signal import butter, sosfilt
from scipy.special import expit
from sklearn.calibration import calibration_curve
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, precision_recall_curve, roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', context='notebook')

@dataclass(frozen=True)
class Config:
    seed: int = 42
    sample_rate: int = 16_000
    crop_seconds: float = 8.0
    crops_per_track: int = 3
    baseline_tracks_per_source: int = 40   # each of human, ACE-Step, MusicGen, Lyria
    challenge_tracks_per_category: int = 5
    train_generators: tuple = ('acestep', 'musicgen')
    unseen_generator: str = 'lyria_pro3'
    target_validation_fpr: float = 0.05
    robustness_tracks_per_class: int = 10
    bootstrap_repeats: int = 300
    cache_dir: str = '/content/haim_guard_cache'
    output_dir: str = '/content/haim_guard_outputs'
    force_recompute_features: bool = False

CFG = Config()
random.seed(CFG.seed)
np.random.seed(CFG.seed)

CACHE_DIR = Path(CFG.cache_dir)
OUTPUT_DIR = Path(CFG.output_dir)
AUDIO_DIR = CACHE_DIR / 'audio'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_HASH = hashlib.sha256(json.dumps(asdict(CFG), sort_keys=True, default=list).encode()).hexdigest()[:12]
print('Configuration hash:', CONFIG_HASH)
print(json.dumps(asdict(CFG), indent=2, default=list))


## Threat model and security objectives

The protected asset is the integrity of music distribution and attribution. The attacker may upload fully generated music while claiming human authorship, impersonate an artist with synthetic vocals, obscure generator traces through mastering or transcoding, splice short AI passages into human recordings, or probe the detector to find transformations that evade it. The detector is assumed to be visible as a black box; its training files and weights are not assumed to be secret.

| Threat | Evaluation in this notebook | Defensive control |
| --- | --- | --- |
| Unseen music generator | Hold out Lyria Pro 3 from all model fitting | Real-only anomaly branch and generator-disjoint test |
| Codec or production laundering | MP3, resampling, EQ/low-pass, gain, clipping, noise, reverb-like filtering | Constrained augmentation, corruption matrix, worst-case reporting |
| Speed/pitch manipulation | Time stretch, resampling, pitch shift | Log-frequency shift-insensitive fingerprint features |
| Hybrid production | HAIM B1–B9 challenge categories | Category-specific sensitivity rather than misleading binary accuracy |
| Partial temporal manipulation | HAIM C1/C2 | Track score plus segment-level extension point |
| False accusation | Target-FPR threshold, calibration, grey-zone abstention | Human review and audit record |
| Dataset leakage | Track-level split; crops never cross partitions | Deterministic split manifest and configuration hash |


In [ ]:
#@title 3. HAIM catalog, role labels, and efficient preview downloader
DATASET_ID = 'mippia/HAIM'
FIRST_ROWS_API = 'https://datasets-server.huggingface.co/first-rows'

# Role labels: 0=human, 1=AI, None=mixed/undefined at whole-track level.
DATASET_SPECS = [
    dict(config='A1_real_MTG_audio_preview', split='mtg', category='A1_real', source='human',
         binary_label=0, composer=0, lyricist=0, vocalist=0, engineer=0, kind='baseline'),
    dict(config='A2_fake_audio_preview', split='acestep', category='A2_full_ai', source='acestep',
         binary_label=1, composer=1, lyricist=1, vocalist=1, engineer=1, kind='baseline'),
    dict(config='A2_fake_audio_preview', split='musicgen', category='A2_full_ai', source='musicgen',
         binary_label=1, composer=1, lyricist=1, vocalist=1, engineer=1, kind='baseline'),
    dict(config='A2_fake_audio_preview', split='lyria_pro3', category='A2_full_ai', source='lyria_pro3',
         binary_label=1, composer=1, lyricist=1, vocalist=1, engineer=1, kind='baseline'),
    dict(config='B_hybrid_audio_preview', split='B1_ai_mastered_human', category='B1_ai_mastered_human', source='hybrid',
         binary_label=None, composer=0, lyricist=0, vocalist=0, engineer=1, kind='hybrid'),
    dict(config='B_hybrid_audio_preview', split='B2_human_mastered_ai_dsp', category='B2_ai_human_ref_mix', source='hybrid',
         binary_label=None, composer=1, lyricist=1, vocalist=1, engineer=1, kind='hybrid'),
    dict(config='B_hybrid_audio_preview', split='B3_mastered', category='B3_ai_human_mastered', source='hybrid',
         binary_label=None, composer=1, lyricist=1, vocalist=1, engineer=0, kind='hybrid'),
    dict(config='B_hybrid_audio_preview', split='B4_professional_human_mix', category='B4_ai_human_mixed', source='hybrid',
         binary_label=None, composer=1, lyricist=1, vocalist=1, engineer=0, kind='hybrid'),
    dict(config='B_hybrid_audio_preview', split='B6_human_lyrics_ai_gen', category='B6_human_lyrics_ai_gen', source='hybrid',
         binary_label=None, composer=1, lyricist=0, vocalist=1, engineer=1, kind='hybrid'),
    dict(config='B_hybrid_audio_preview', split='B7_variation', category='B7_ai_variation', source='hybrid',
         binary_label=None, composer=1, lyricist=1, vocalist=1, engineer=1, kind='hybrid'),
    dict(config='B_hybrid_audio_preview', split='B8_edit', category='B8_ai_edit', source='hybrid',
         binary_label=None, composer=1, lyricist=1, vocalist=1, engineer=1, kind='hybrid'),
    dict(config='B_hybrid_audio_preview', split='B9_repaint', category='B9_ai_repaint', source='hybrid',
         binary_label=None, composer=1, lyricist=1, vocalist=1, engineer=1, kind='hybrid'),
    dict(config='C_mixing_audio_preview', split='C1_concat', category='C1_temporal_concat', source='temporal_mix',
         binary_label=None, composer=None, lyricist=None, vocalist=None, engineer=None, kind='temporal'),
    dict(config='C_mixing_audio_preview', split='C2_crossfade', category='C2_temporal_crossfade', source='temporal_mix',
         binary_label=None, composer=None, lyricist=None, vocalist=None, engineer=None, kind='temporal'),
]


def _audio_url_from_cell(cell):
    if isinstance(cell, list) and cell:
        return cell[0].get('src'), cell[0].get('type', 'audio/mpeg')
    if isinstance(cell, dict):
        return cell.get('src') or cell.get('path'), cell.get('type', 'audio/mpeg')
    raise ValueError(f'Unexpected audio cell: {type(cell)}')


def fetch_preview_split(spec, limit, retries=3):
    """Download only the requested first rows; avoids fetching a 245 GB corpus."""
    params = {'dataset': DATASET_ID, 'config': spec['config'], 'split': spec['split']}
    response = requests.get(FIRST_ROWS_API, params=params, timeout=90)
    response.raise_for_status()
    rows = response.json()['rows'][:limit]
    records = []

    out_dir = AUDIO_DIR / spec['split']
    out_dir.mkdir(parents=True, exist_ok=True)

    for item in tqdm(rows, desc=spec['split'], leave=False):
        row = item['row']
        url, mime = _audio_url_from_cell(row['audio'])
        track_id = str(row.get('track_id', row.get('id', item['row_idx'])))
        ext = '.wav' if 'wav' in mime else '.mp3'
        local_path = out_dir / f"{item['row_idx']:04d}_{track_id.replace('/', '_')}{ext}"

        if not local_path.exists() or local_path.stat().st_size == 0:
            last_error = None
            for attempt in range(retries):
                try:
                    with requests.get(url, stream=True, timeout=180) as audio_response:
                        audio_response.raise_for_status()
                        with open(local_path, 'wb') as handle:
                            for chunk in audio_response.iter_content(chunk_size=1024 * 1024):
                                if chunk:
                                    handle.write(chunk)
                    break
                except Exception as exc:
                    last_error = exc
                    time.sleep(2 ** attempt)
            else:
                print(f"Skipping {spec['split']} row {item['row_idx']}: {last_error}")
                continue

        metadata = {k: v for k, v in row.items() if k != 'audio'}
        record = {**spec, **metadata}
        record.update(
            record_id=f"{spec['split']}::{track_id}",
            audio_path=str(local_path),
            row_idx=item['row_idx'],
        )
        records.append(record)
    return records


def download_quick_dataset():
    all_records = []
    for spec in DATASET_SPECS:
        n = CFG.baseline_tracks_per_source if spec['kind'] == 'baseline' else CFG.challenge_tracks_per_category
        all_records.extend(fetch_preview_split(spec, n))
    manifest = pd.DataFrame(all_records)
    manifest.to_csv(CACHE_DIR / f'manifest_{CONFIG_HASH}.csv', index=False)
    return manifest

manifest_path = CACHE_DIR / f'manifest_{CONFIG_HASH}.csv'
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
else:
    manifest = download_quick_dataset()

print(f'Downloaded/cached {len(manifest):,} tracks.')
display(manifest.groupby(['kind', 'category', 'source'], dropna=False).size().rename('tracks').reset_index())


### Optional Research Mode

The cell below is intentionally disabled. It shows the official supported method for downloading selected raw folders. **Do not remove `allow_patterns` unless you have approximately 250 GB of free disk.** The preview loader above is the recommended default.


In [ ]:
#@title 4. Optional: download selected full HAIM folders (disabled by default)
RUN_RESEARCH_MODE_DOWNLOAD = False

if RUN_RESEARCH_MODE_DOWNLOAD:
    from huggingface_hub import snapshot_download
    selected_path = snapshot_download(
        repo_id=DATASET_ID,
        repo_type='dataset',
        local_dir='/content/HAIM_selected',
        allow_patterns=[
            'A_full_generation/A1_real/MTG-Jamendo music subset/**',
            'A_full_generation/A2_fake/musicgen/**',
            # Add only the categories you can store, for example:
            # 'B_hybrid/B1*/**',
            # 'C_mixing/C1*/**',
        ],
    )
    print('Selected raw folders:', selected_path)
else:
    print('Research Mode is disabled. Quick Mode uses the official preview audio only.')


In [ ]:
#@title 5. Dataset audit and leakage-resistant split

def deterministic_three_way(ids, seed, ratios=(0.50, 0.25, 0.25)):
    ids = np.asarray(sorted(ids))
    rng = np.random.default_rng(seed)
    shuffled = ids[rng.permutation(len(ids))]
    n_train = int(round(len(ids) * ratios[0]))
    n_val = int(round(len(ids) * ratios[1]))
    assignment = {}
    for rid in shuffled[:n_train]: assignment[rid] = 'train'
    for rid in shuffled[n_train:n_train+n_val]: assignment[rid] = 'val'
    for rid in shuffled[n_train+n_val:]: assignment[rid] = 'id_test'
    return assignment

manifest['partition'] = 'challenge'
for source in ['human', *CFG.train_generators]:
    idx = manifest['source'].eq(source)
    mapping = deterministic_three_way(manifest.loc[idx, 'record_id'].tolist(), CFG.seed + len(source))
    manifest.loc[idx, 'partition'] = manifest.loc[idx, 'record_id'].map(mapping)
manifest.loc[manifest['source'].eq(CFG.unseen_generator), 'partition'] = 'ood_test'

# Integrity checks: every track is unique, no track is split twice, files exist.
assert manifest['record_id'].is_unique, 'Duplicate track IDs would invalidate group-safe evaluation.'
assert manifest['audio_path'].map(os.path.exists).all(), 'One or more cached audio files are missing.'
assert set(CFG.train_generators).isdisjoint({CFG.unseen_generator})

split_audit = manifest.groupby(['partition', 'source', 'category'], dropna=False).size().rename('tracks').reset_index()
display(split_audit)

plt.figure(figsize=(11, 4))
sns.countplot(data=manifest, x='category', hue='partition')
plt.xticks(rotation=70, ha='right')
plt.title('HAIM-Guard split audit (tracks, never crops)')
plt.tight_layout()
plt.show()


## Multiview forensic representation

The representation combines interpretable time–frequency statistics with a compact shift-insensitive fingerprint. The latter maps the time-averaged spectrum onto a logarithmic frequency axis, removes a smooth baseline, and takes the magnitude of a one-dimensional Fourier transform. A translation on the log-frequency axis changes phase but preserves Fourier magnitude, so these features are less sensitive to global frequency scaling. This is an educational approximation of the stronger invariance-by-construction method published in 2026; it is **not** presented as an exact reproduction.[8]

Each track produces three deterministic 8-second crops. Crops from one track always remain in the same data partition, and probabilities are averaged back to the track level.


In [ ]:
#@title 6. Audio loading, deterministic crops, and feature extraction

def load_audio(path, sr=CFG.sample_rate):
    y, _ = librosa.load(path, sr=sr, mono=True, dtype=np.float32)
    y = np.nan_to_num(y)
    peak = np.max(np.abs(y)) if len(y) else 0.0
    if peak > 1.0:
        y = y / peak
    return y


def crop_positions(n_samples, crop_samples, n_crops=3):
    if n_samples <= crop_samples:
        return [0] * n_crops
    max_start = n_samples - crop_samples
    fractions = np.linspace(0.20, 0.80, n_crops)
    return [int(round(max_start * f)) for f in fractions]


def fixed_crops(y, crop_seconds=CFG.crop_seconds, n_crops=CFG.crops_per_track, sr=CFG.sample_rate):
    length = int(crop_seconds * sr)
    if len(y) < length:
        y = np.pad(y, (0, length - len(y)))
    starts = crop_positions(len(y), length, n_crops)
    return [y[s:s+length].astype(np.float32, copy=False) for s in starts]


def training_augment(y, record_id, sr=CFG.sample_rate):
    """One deterministic, content-preserving augmentation per training track."""
    seed = int(hashlib.sha256(str(record_id).encode()).hexdigest()[:8], 16) + CFG.seed
    rng = np.random.default_rng(seed)
    operation = seed % 4
    if operation == 0:  # gain
        return (y * rng.uniform(0.55, 0.90)).astype(np.float32)
    if operation == 1:  # light additive noise, 24–32 dB SNR
        signal_rms = np.sqrt(np.mean(y**2) + 1e-9)
        snr_db = rng.uniform(24.0, 32.0)
        noise_rms = signal_rms / (10 ** (snr_db / 20))
        return (y + rng.normal(0, noise_rms, size=len(y))).astype(np.float32)
    if operation == 2:  # sample-rate round trip
        mid_sr = 12_000
        z = librosa.resample(y, orig_sr=sr, target_sr=mid_sr)
        return librosa.resample(z, orig_sr=mid_sr, target_sr=sr).astype(np.float32)
    sos = butter(4, 6000, btype='low', fs=sr, output='sos')
    return sosfilt(sos, y).astype(np.float32)


def summarize_matrix(matrix, prefix, names, values):
    matrix = np.asarray(matrix)
    means = np.mean(matrix, axis=1)
    stds = np.std(matrix, axis=1)
    for i, value in enumerate(means):
        names.append(f'{prefix}_mean_{i:03d}'); values.append(float(value))
    for i, value in enumerate(stds):
        names.append(f'{prefix}_std_{i:03d}'); values.append(float(value))


def log_frequency_invariant_features(y, sr, n_bins=128, n_keep=65):
    # A compact, shift-insensitive spectral fingerprint.
    mag = np.abs(librosa.stft(y, n_fft=4096, hop_length=512, win_length=4096))
    spectrum = np.mean(mag, axis=1)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=4096)
    f_min, f_max = 60.0, min(sr / 2 - 1, 7600.0)
    log_freqs = np.geomspace(f_min, f_max, n_bins)
    interp = np.interp(log_freqs, freqs, spectrum)
    log_spec = np.log1p(interp)
    baseline = median_filter(log_spec, size=17, mode='nearest')
    residual = np.maximum(log_spec - baseline, 0.0)
    residual = (residual - residual.mean()) / (residual.std() + 1e-6)
    invariant = np.log1p(np.abs(np.fft.rfft(residual)))[:n_keep]
    return invariant.astype(np.float32)


def extract_features(y, sr=CFG.sample_rate):
    eps = 1e-8
    names, values = [], []

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=1024, hop_length=256, n_mels=64, fmin=40, fmax=sr/2,
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel + eps, ref=np.max)
    summarize_matrix(mel_db, 'mel', names, values)

    mfcc = librosa.feature.mfcc(S=mel_db, n_mfcc=20)
    summarize_matrix(mfcc, 'mfcc', names, values)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_fft=2048, hop_length=512)
    summarize_matrix(chroma, 'chroma', names, values)

    scalar_series = {
        'centroid': librosa.feature.spectral_centroid(y=y, sr=sr),
        'bandwidth': librosa.feature.spectral_bandwidth(y=y, sr=sr),
        'rolloff': librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85),
        'flatness': librosa.feature.spectral_flatness(y=y),
        'zcr': librosa.feature.zero_crossing_rate(y),
        'rms': librosa.feature.rms(y=y),
    }
    for key, arr in scalar_series.items():
        names.extend([f'{key}_mean', f'{key}_std'])
        values.extend([float(np.mean(arr)), float(np.std(arr))])

    fakeprint = log_frequency_invariant_features(y, sr)
    for i, value in enumerate(fakeprint):
        names.append(f'logfreq_fft_{i:03d}'); values.append(float(value))

    vector = np.nan_to_num(np.asarray(values, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return vector, names

# Smoke test one file.
_test_audio = load_audio(manifest.iloc[0]['audio_path'])
_test_vector, FEATURE_NAMES = extract_features(fixed_crops(_test_audio)[0])
print('Feature dimension:', len(_test_vector))
print('First features:', FEATURE_NAMES[:8])


In [ ]:
#@title 7. Extract and cache crop-level features
feature_cache = CACHE_DIR / f'features_{CONFIG_HASH}.parquet'

if feature_cache.exists() and not CFG.force_recompute_features:
    features = pd.read_parquet(feature_cache)
else:
    rows = []
    for record in tqdm(manifest.to_dict('records'), desc='Tracks'):
        try:
            y = load_audio(record['audio_path'])
            crops = fixed_crops(y)
            crop_items = list(enumerate(crops))
            if record['partition'] == 'train' and record['kind'] == 'baseline':
                crop_items.append((100, training_augment(crops[len(crops)//2], record['record_id'])))
            for crop_idx, crop in crop_items:
                vector, names = extract_features(crop)
                row = {
                    'record_id': record['record_id'],
                    'audio_path': record['audio_path'],
                    'partition': record['partition'],
                    'source': record['source'],
                    'category': record['category'],
                    'kind': record['kind'],
                    'binary_label': record.get('binary_label'),
                    'composer': record.get('composer'),
                    'lyricist': record.get('lyricist'),
                    'vocalist': record.get('vocalist'),
                    'engineer': record.get('engineer'),
                    'crop_idx': crop_idx,
                    'augmented': crop_idx >= 100,
                }
                row.update({f'f_{name}': value for name, value in zip(names, vector)})
                rows.append(row)
        except Exception as exc:
            print('Feature extraction failed:', record['record_id'], exc)
    features = pd.DataFrame(rows)
    features.to_parquet(feature_cache, index=False)

FEATURE_COLS = [c for c in features.columns if c.startswith('f_')]
print(f'{len(features):,} crop rows, {features.record_id.nunique():,} tracks, {len(FEATURE_COLS):,} features')
assert features.groupby('record_id')['partition'].nunique().max() == 1
assert features[FEATURE_COLS].notna().all().all()


In [ ]:
#@title 8. Exploratory audit: do obvious dataset artifacts dominate?
from sklearn.manifold import TSNE

track_features = features.groupby(
    ['record_id', 'partition', 'source', 'category', 'kind', 'binary_label'], dropna=False
)[FEATURE_COLS].mean().reset_index()

sample_for_plot = track_features.sample(min(len(track_features), 250), random_state=CFG.seed)
X_plot = StandardScaler().fit_transform(sample_for_plot[FEATURE_COLS])
X_plot = PCA(n_components=min(20, X_plot.shape[0]-1, X_plot.shape[1]), random_state=CFG.seed).fit_transform(X_plot)
embedding = TSNE(n_components=2, perplexity=min(25, max(5, len(X_plot)//8)), init='pca', learning_rate='auto', random_state=CFG.seed).fit_transform(X_plot)

plot_df = sample_for_plot[['source', 'category', 'partition']].copy()
plot_df['x'] = embedding[:, 0]
plot_df['y'] = embedding[:, 1]
plt.figure(figsize=(10, 6))
sns.scatterplot(data=plot_df, x='x', y='y', hue='source', style='partition', s=70, alpha=.8)
plt.title('Feature-space audit (t-SNE is descriptive, not evidence of generalization)')
plt.tight_layout()
plt.show()

print('Check whether clusters align almost perfectly with file source or category. If so, the model may be exploiting collection artifacts.')


## Model design

The primary score fuses two complementary branches. The supervised branch averages a transparent logistic regression and a nonlinear histogram gradient-boosting model. The second branch is fit only on human training tracks: standardized features are projected with PCA and modeled using shrinkage covariance; large Mahalanobis distance indicates an anomaly. This branch follows MusicDET's **real-distribution modeling principle** but is intentionally smaller and is not an implementation of MusicDET's normalizing flows.[7]

The supervised models train on crops, but calibration, thresholds, and all reported metrics operate at the **track level**.


In [ ]:
#@title 9. Train supervised and real-only branches
train_rows = features[features['partition'].eq('train') & features['binary_label'].notna()].copy()
val_rows = features[features['partition'].eq('val') & features['binary_label'].notna()].copy()

X_train = train_rows[FEATURE_COLS].to_numpy(np.float32)
y_train = train_rows['binary_label'].astype(int).to_numpy()
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)

lr_model = Pipeline([
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(C=0.35, max_iter=4000, class_weight='balanced', random_state=CFG.seed)),
])
lr_model.fit(X_train, y_train)

hgb_model = HistGradientBoostingClassifier(
    learning_rate=0.06, max_iter=260, max_leaf_nodes=15,
    min_samples_leaf=8, l2_regularization=1.5, random_state=CFG.seed,
)
hgb_model.fit(X_train, y_train, sample_weight=sample_weight)

# Real-only branch: fit no fake music into the representation/density model.
real_train = train_rows[train_rows['binary_label'].eq(0)][FEATURE_COLS].to_numpy(np.float32)
real_scaler = StandardScaler().fit(real_train)
real_train_scaled = real_scaler.transform(real_train)
n_components = max(2, min(12, real_train_scaled.shape[0] - 2, real_train_scaled.shape[1]))
real_pca = PCA(n_components=n_components, whiten=True, random_state=CFG.seed).fit(real_train_scaled)
real_train_z = real_pca.transform(real_train_scaled)
real_cov = LedoitWolf().fit(real_train_z)


def anomaly_distance(X):
    z = real_pca.transform(real_scaler.transform(X))
    return real_cov.mahalanobis(z)


def raw_crop_scores(df):
    X = df[FEATURE_COLS].to_numpy(np.float32)
    out = df[['record_id', 'partition', 'source', 'category', 'kind', 'binary_label', 'crop_idx']].copy()
    out['p_lr'] = lr_model.predict_proba(X)[:, 1]
    out['p_hgb'] = hgb_model.predict_proba(X)[:, 1]
    out['p_supervised_raw'] = 0.5 * (out['p_lr'] + out['p_hgb'])
    out['anomaly_distance'] = anomaly_distance(X)
    return out

val_crop_scores = raw_crop_scores(val_rows)
val_track = val_crop_scores.groupby(
    ['record_id', 'partition', 'source', 'category', 'kind', 'binary_label'], dropna=False
).mean(numeric_only=True).reset_index()

# Calibration is fit on held-out tracks, never on their training crops.
iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
if val_track['p_supervised_raw'].nunique() >= 4 and val_track['binary_label'].nunique() == 2:
    iso.fit(val_track['p_supervised_raw'], val_track['binary_label'].astype(int))
else:
    raise RuntimeError('Validation set is too small for calibration; increase baseline_tracks_per_source.')

real_val_dist = np.sort(val_track.loc[val_track['binary_label'].eq(0), 'anomaly_distance'].to_numpy())

def anomaly_probability(distances):
    # Empirical real-only tail probability; high distance -> high anomaly probability.
    ranks = np.searchsorted(real_val_dist, np.asarray(distances), side='right')
    return np.clip(ranks / max(1, len(real_val_dist)), 0.0, 1.0)


def finalize_track_scores(crop_score_df):
    track = crop_score_df.groupby(
        ['record_id', 'partition', 'source', 'category', 'kind', 'binary_label'], dropna=False
    ).mean(numeric_only=True).reset_index()
    track['p_supervised'] = iso.predict(track['p_supervised_raw'])
    track['p_anomaly'] = anomaly_probability(track['anomaly_distance'])
    track['score'] = 0.75 * track['p_supervised'] + 0.25 * track['p_anomaly']
    return track

val_track = finalize_track_scores(val_crop_scores)
display(val_track[['record_id', 'source', 'binary_label', 'p_supervised', 'p_anomaly', 'score']].head())


In [ ]:
#@title 10. False-positive-constrained thresholds and abstention policy

def select_threshold_at_fpr(y_true, scores, target_fpr=0.05):
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    valid = np.where(fpr <= target_fpr)[0]
    if len(valid) == 0:
        return float(np.quantile(scores[np.asarray(y_true) == 0], 1 - target_fpr))
    best = valid[np.argmax(tpr[valid])]
    return float(thresholds[best])

val_y = val_track['binary_label'].astype(int).to_numpy()
val_scores = val_track['score'].to_numpy()
binary_threshold = select_threshold_at_fpr(val_y, val_scores, CFG.target_validation_fpr)

human_q95 = float(np.quantile(val_track.loc[val_track.binary_label.eq(0), 'score'], 0.95))
fake_q10 = float(np.quantile(val_track.loc[val_track.binary_label.eq(1), 'score'], 0.10))
review_low = min(binary_threshold, human_q95)
review_high = max(binary_threshold, fake_q10)
if review_high - review_low < 0.05:
    review_low = max(0.0, binary_threshold - 0.025)
    review_high = min(1.0, binary_threshold + 0.025)

THRESHOLDS = {
    'binary_threshold': binary_threshold,
    'review_low': review_low,
    'review_high': review_high,
    'target_validation_fpr': CFG.target_validation_fpr,
}
print(json.dumps(THRESHOLDS, indent=2))


def policy_label(score):
    if score < review_low:
        return 'likely_human'
    if score >= review_high:
        return 'ai_indicators_detected'
    return 'uncertain_human_review'

val_track['policy'] = val_track['score'].map(policy_label)
display(pd.crosstab(val_track['binary_label'], val_track['policy'], normalize='index').round(3))


In [ ]:
#@title 11. Security-focused metrics, EER, calibration error, and bootstrap intervals

def equal_error_rate(y_true, scores):
    fpr, tpr, _ = roc_curve(y_true, scores)
    fnr = 1 - tpr
    idx = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[idx] + fnr[idx]) / 2)


def expected_calibration_error(y_true, scores, n_bins=10):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (scores >= lo) & (scores < hi if hi < 1 else scores <= hi)
        if mask.any():
            ece += mask.mean() * abs(scores[mask].mean() - y_true[mask].mean())
    return float(ece)


def fpr_at_tpr(y_true, scores, target_tpr=0.90):
    fpr, tpr, _ = roc_curve(y_true, scores)
    valid = np.where(tpr >= target_tpr)[0]
    return float(np.min(fpr[valid])) if len(valid) else float('nan')


def metric_row(y_true, scores, threshold, name):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)
    pred = (scores >= threshold).astype(int)
    return {
        'evaluation': name,
        'n_tracks': len(y_true),
        'roc_auc': roc_auc_score(y_true, scores),
        'pr_auc': average_precision_score(y_true, scores),
        'f1': f1_score(y_true, pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_true, pred),
        'eer': equal_error_rate(y_true, scores),
        'brier': brier_score_loss(y_true, scores),
        'ece': expected_calibration_error(y_true, scores),
        'fpr_at_tpr90': fpr_at_tpr(y_true, scores),
    }


def bootstrap_ci(y_true, scores, metric_fn, repeats=CFG.bootstrap_repeats):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    rng = np.random.default_rng(CFG.seed)
    values = []
    for _ in range(repeats):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        values.append(metric_fn(y_true[idx], scores[idx]))
    return tuple(np.quantile(values, [0.025, 0.975])) if values else (np.nan, np.nan)


In [ ]:
#@title 12. In-domain and unseen-generator evaluation
all_crop_scores = raw_crop_scores(features)
all_track_scores = finalize_track_scores(all_crop_scores)
all_track_scores['policy'] = all_track_scores['score'].map(policy_label)

human_test_ids = set(manifest.loc[(manifest.source.eq('human')) & (manifest.partition.eq('id_test')), 'record_id'])

id_eval = all_track_scores[
    all_track_scores['partition'].eq('id_test') & all_track_scores['binary_label'].notna()
].copy()

# OOD evaluation reuses held-out human tracks and a balanced sample from the never-trained generator.
human_ood = all_track_scores[all_track_scores.record_id.isin(human_test_ids)]
unseen_all = all_track_scores[all_track_scores['source'].eq(CFG.unseen_generator)]
unseen_balanced = unseen_all.sample(n=min(len(human_ood), len(unseen_all)), random_state=CFG.seed)
ood_eval = pd.concat([human_ood, unseen_balanced], ignore_index=True)

# Evaluate each generator against the same held-out human tracks. These are not independent tests,
# but they expose source-specific failure rather than hiding it in an average.
generator_frames = []
for generator in [*CFG.train_generators, CFG.unseen_generator]:
    fake_group = all_track_scores[(all_track_scores.source.eq(generator)) & all_track_scores.partition.isin(['id_test', 'ood_test'])]
    fake_group = fake_group.sample(n=min(len(human_ood), len(fake_group)), random_state=CFG.seed)
    group_eval = pd.concat([human_ood, fake_group], ignore_index=True)
    generator_frames.append(metric_row(group_eval.binary_label, group_eval.score, binary_threshold, f'Generator: {generator}'))

clean_metrics = pd.DataFrame([
    metric_row(id_eval.binary_label, id_eval.score, binary_threshold, 'ID: seen generators'),
    metric_row(ood_eval.binary_label, ood_eval.score, binary_threshold, f'OOD: unseen {CFG.unseen_generator}'),
    *generator_frames,
])

for frame, name in [(id_eval, 'ID'), (ood_eval, 'OOD')]:
    lo, hi = bootstrap_ci(frame.binary_label.astype(int), frame.score, roc_auc_score)
    clean_metrics.loc[clean_metrics.evaluation.str.startswith(name), 'roc_auc_ci95'] = f'[{lo:.3f}, {hi:.3f}]'

display(clean_metrics.round(4))
clean_metrics.to_csv(OUTPUT_DIR / 'metrics_clean.csv', index=False)
all_track_scores.to_csv(OUTPUT_DIR / 'predictions.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for frame, name, color in [(id_eval, 'ID', '#1f77b4'), (ood_eval, 'Unseen generator', '#d62728')]:
    fpr, tpr, _ = roc_curve(frame.binary_label.astype(int), frame.score)
    axes[0].plot(fpr, tpr, label=f'{name} AUC={roc_auc_score(frame.binary_label, frame.score):.3f}', color=color)
    precision, recall, _ = precision_recall_curve(frame.binary_label.astype(int), frame.score)
    axes[1].plot(recall, precision, label=f'{name} AP={average_precision_score(frame.binary_label, frame.score):.3f}', color=color)
axes[0].plot([0, 1], [0, 1], '--', color='grey'); axes[0].set(title='ROC', xlabel='False-positive rate', ylabel='True-positive rate'); axes[0].legend()
axes[1].set(title='Precision–recall', xlabel='Recall', ylabel='Precision'); axes[1].legend()
prob_true, prob_pred = calibration_curve(id_eval.binary_label.astype(int), id_eval.score, n_bins=6, strategy='quantile')
axes[2].plot(prob_pred, prob_true, marker='o', label='ID calibration')
axes[2].plot([0, 1], [0, 1], '--', color='grey'); axes[2].set(title='Calibration', xlabel='Mean score', ylabel='Observed AI rate'); axes[2].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'clean_roc_pr_calibration.png', dpi=170, bbox_inches='tight')
plt.show()


In [ ]:
#@title 13. HAIM hybrid and temporal challenge evaluation
challenge_scores = all_track_scores[all_track_scores['kind'].isin(['hybrid', 'temporal'])].copy()
challenge_scores['ai_alert'] = challenge_scores['score'] >= review_high
challenge_scores['uncertain'] = challenge_scores['policy'].eq('uncertain_human_review')

hybrid_metrics = challenge_scores.groupby(['kind', 'category']).agg(
    n_tracks=('record_id', 'nunique'),
    mean_score=('score', 'mean'),
    median_score=('score', 'median'),
    ai_alert_rate=('ai_alert', 'mean'),
    uncertain_rate=('uncertain', 'mean'),
).reset_index()

display(hybrid_metrics.round(3))
hybrid_metrics.to_csv(OUTPUT_DIR / 'metrics_hybrid.csv', index=False)

plt.figure(figsize=(11, 5))
order = hybrid_metrics.sort_values('mean_score').category
sns.boxplot(data=challenge_scores, y='category', x='score', order=order, color='#8dd3c7')
plt.axvline(review_low, color='#f0ad4e', linestyle='--', label='review low')
plt.axvline(review_high, color='#d9534f', linestyle='--', label='AI alert')
plt.title('Scores on hybrid and temporally mixed HAIM categories')
plt.xlabel('Calibrated HAIM-Guard score')
plt.ylabel('')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'hybrid_category_scores.png', dpi=170, bbox_inches='tight')
plt.show()

print('Interpretation warning: these are sensitivity rates, not binary accuracy. HAIM hybrid tracks have role-level labels.')


## Robustness and laundering evaluation

The corruption suite applies transformations to **both** real and synthetic tracks. Applying a transform only to fakes would create a trivial shortcut. The suite contains observed-in-training perturbations and held-out transformations. Results should be reported as absolute performance and degradation relative to clean audio. A security claim should emphasize the **worst corruption**, not the average.


In [ ]:
#@title 14. Corruption functions

def crop_or_pad(y, n):
    if len(y) >= n:
        start = (len(y) - n) // 2
        return y[start:start+n]
    return np.pad(y, (0, n-len(y)))


def resample_roundtrip(y, sr, intermediate_sr=12_000):
    return librosa.resample(librosa.resample(y, orig_sr=sr, target_sr=intermediate_sr), orig_sr=intermediate_sr, target_sr=sr)


def add_noise(y, snr_db=20.0, rng=None):
    rng = rng or np.random.default_rng(CFG.seed)
    signal_rms = np.sqrt(np.mean(y**2) + 1e-9)
    noise_rms = signal_rms / (10 ** (snr_db / 20))
    return y + rng.normal(0, noise_rms, size=len(y)).astype(np.float32)


def lowpass(y, sr, cutoff=3400):
    sos = butter(6, cutoff, btype='low', fs=sr, output='sos')
    return sosfilt(sos, y).astype(np.float32)


def mp3_roundtrip(y, sr, bitrate='64k'):
    with tempfile.TemporaryDirectory() as td:
        wav_path = Path(td) / 'in.wav'
        mp3_path = Path(td) / 'out.mp3'
        sf.write(wav_path, y, sr)
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(wav_path), '-b:a', bitrate, str(mp3_path)], check=True)
        decoded, _ = librosa.load(mp3_path, sr=sr, mono=True, dtype=np.float32)
    return decoded


def apply_corruption(y, name, sr=CFG.sample_rate, seed=CFG.seed):
    rng = np.random.default_rng(seed)
    if name == 'clean': return y.copy()
    if name == 'gain_-6db': return (0.5 * y).astype(np.float32)
    if name == 'noise_20db': return add_noise(y, 20.0, rng)
    if name == 'resample_12k': return resample_roundtrip(y, sr, 12_000).astype(np.float32)
    if name == 'lowpass_3.4k': return lowpass(y, sr, 3400)
    if name == 'clip_0.30': return np.clip(y, -0.30, 0.30).astype(np.float32)
    if name == 'time_shift': return np.roll(y, int(0.20 * sr)).astype(np.float32)
    if name == 'time_stretch_1.08': return librosa.effects.time_stretch(y, rate=1.08).astype(np.float32)
    if name == 'pitch_shift_+1': return librosa.effects.pitch_shift(y, sr=sr, n_steps=1.0).astype(np.float32)
    if name == 'mp3_64k': return mp3_roundtrip(y, sr, '64k')
    if name == 'combined':
        z = resample_roundtrip(y, sr, 12_000)
        z = add_noise(z, 24.0, rng)
        return np.clip(0.7 * lowpass(z, sr, 5200), -0.8, 0.8).astype(np.float32)
    raise KeyError(name)

CORRUPTIONS = [
    'clean', 'gain_-6db', 'noise_20db', 'resample_12k', 'lowpass_3.4k',
    'clip_0.30', 'time_shift', 'time_stretch_1.08', 'pitch_shift_+1',
    'mp3_64k', 'combined',
]
print(CORRUPTIONS)


In [ ]:
#@title 15. Run the corruption benchmark on balanced unseen-generator tracks
human_robust = manifest[(manifest.source.eq('human')) & (manifest.partition.eq('id_test'))].head(CFG.robustness_tracks_per_class)
fake_robust = manifest[manifest.source.eq(CFG.unseen_generator)].head(CFG.robustness_tracks_per_class)
robust_manifest = pd.concat([human_robust, fake_robust], ignore_index=True)


def score_feature_matrix(X):
    p_lr = lr_model.predict_proba(X)[:, 1]
    p_hgb = hgb_model.predict_proba(X)[:, 1]
    p_sup = iso.predict(0.5 * (p_lr + p_hgb))
    p_anom = anomaly_probability(anomaly_distance(X))
    return 0.75 * p_sup + 0.25 * p_anom

robust_rows = []
segment_samples = int(30 * CFG.sample_rate)
for record in tqdm(robust_manifest.to_dict('records'), desc='Robustness tracks'):
    full = load_audio(record['audio_path'])
    base = crop_or_pad(full, segment_samples)
    for corruption in CORRUPTIONS:
        try:
            altered = apply_corruption(base, corruption, seed=CFG.seed + int(record.get('row_idx', 0)))
            altered = crop_or_pad(altered, segment_samples)
            crop_vectors = np.vstack([extract_features(c)[0] for c in fixed_crops(altered)])
            score = float(np.mean(score_feature_matrix(crop_vectors)))
            robust_rows.append({
                'record_id': record['record_id'], 'source': record['source'],
                'binary_label': int(record['binary_label']), 'corruption': corruption,
                'score': score,
            })
        except Exception as exc:
            print('Skipping corruption:', corruption, record['record_id'], exc)

robust_predictions = pd.DataFrame(robust_rows)
robust_metrics = pd.DataFrame([
    metric_row(group.binary_label, group.score, binary_threshold, corruption)
    for corruption, group in robust_predictions.groupby('corruption')
])
clean_f1 = float(robust_metrics.loc[robust_metrics.evaluation.eq('clean'), 'f1'].iloc[0])
robust_metrics['f1_drop_vs_clean'] = clean_f1 - robust_metrics['f1']
robust_metrics = robust_metrics.sort_values('f1')
display(robust_metrics.round(4))
robust_metrics.to_csv(OUTPUT_DIR / 'metrics_corruptions.csv', index=False)
robust_predictions.to_csv(OUTPUT_DIR / 'predictions_corruptions.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
sns.barplot(data=robust_metrics, y='evaluation', x='f1', color='#80b1d3', ax=axes[0])
axes[0].axvline(clean_f1, color='black', linestyle='--', label=f'clean F1={clean_f1:.3f}')
axes[0].set(xlim=(0, 1), title='Thresholded robustness', xlabel='F1', ylabel='')
axes[0].legend()
sns.barplot(data=robust_metrics, y='evaluation', x='roc_auc', color='#fb8072', ax=axes[1])
axes[1].set(xlim=(0, 1), title='Threshold-independent robustness', xlabel='ROC-AUC', ylabel='')
fig.suptitle('Corruption robustness on an unseen generator (worst F1 first)', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'corruption_f1.png', dpi=170, bbox_inches='tight')
plt.show()


## Optional same-sample benchmark with the official SpecTTTra checkpoint

The next cell evaluates the public SONICS checkpoint on local HAIM test files. This is more informative than comparing unrelated literature numbers, although it still reflects a domain shift: SpecTTTra was trained on SONICS, not HAIM. The default 5-second checkpoint is small enough for Colab. Set `RUN_OFFICIAL_SPECTTTRA=True` to execute it. The published SONICS F1 scores are 0.78 for the 5-second alpha/beta checkpoints and 0.97 for the 120-second alpha checkpoint.[3]


In [ ]:
#@title 16. Optional public SOTA adapter: official SONICS SpecTTTra
RUN_OFFICIAL_SPECTTTRA = False
SPECTTTRA_MODEL_ID = 'awsaf49/sonics-spectttra-alpha-5s'

if RUN_OFFICIAL_SPECTTTRA:
    %pip -q install git+https://github.com/awsaf49/sonics.git
    import torch
    from sonics import HFAudioClassifier

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    spectttra = HFAudioClassifier.from_pretrained(SPECTTTRA_MODEL_ID).to(device).eval()

    def spectttra_score(path):
        y, _ = librosa.load(path, sr=16_000, mono=True, dtype=np.float32)
        n = 5 * 16_000
        y = crop_or_pad(y, n)
        y = y / max(np.std(y), 1e-6)
        with torch.inference_mode():
            logits = spectttra(torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(0))
            return float(torch.sigmoid(logits.reshape(-1)[0]).cpu())

    sota_subset = pd.concat([
        manifest[(manifest.source.eq('human')) & (manifest.partition.eq('id_test'))],
        manifest[manifest.source.eq(CFG.unseen_generator)],
    ], ignore_index=True)
    sota_subset['spectttra_score'] = [spectttra_score(p) for p in tqdm(sota_subset.audio_path)]
    sota_metric = metric_row(
        sota_subset.binary_label.astype(int), sota_subset.spectttra_score,
        threshold=0.5, name='Official SpecTTTra-α-5s on HAIM unseen-generator set',
    )
    display(pd.DataFrame([sota_metric]).round(4))
    sota_subset.to_csv(OUTPUT_DIR / 'predictions_spectttra.csv', index=False)
else:
    print('Skipped. Set RUN_OFFICIAL_SPECTTTRA=True to download the MIT-licensed public checkpoint and run it.')


## Published SOTA reference benchmark

The table below is a **literature comparison, not a leaderboard**. Segment durations, datasets, train/test protocols, and metrics differ. The most security-relevant number is usually the generator-disjoint or attacked-audio result, not the largest in-domain score.

| Public model | Published setting | Reported result | Security interpretation |
| --- | --- | --- | --- |
| **MusicDET, real-only** | FakeMusicCaps cross-generator | Average EER **4.51%** | Strong current zero-shot result; training uses only real music.[7] |
| **Class-conditional MusicDET** | FakeMusicCaps cross-generator | Average EER **0.89%** | Uses fake examples; excellent cross-generator result, but not zero-shot.[7] |
| **MusicDET, real-only** | SONICS cross-generator | Average EER **2.89%** | Four-second protocol; strong but not directly comparable to full-song F1.[7] |
| **CLAM** | MoM generator-disjoint test | Accuracy **93.1%**, F1 **92.5%** | Strong OOD benchmark with multiple unseen generators.[4] |
| **CLAM** | SONICS | F1 **99.3%** | Near saturation on an in-domain benchmark.[4] |
| **SpecTTTra-α** | SONICS, 120 seconds | F1 **97.2%** | Efficient long-context benchmark; MoM reports large drops by unseen generator.[3] [4] |
| **Fusion Segment Transformer (MERT)** | SONICS | F1 **99.99%** | Authors warn that SONICS resampling may create an easy shortcut.[5] |
| **Fusion Segment Transformer (MERT)** | AIME | F1 **98.68%**, AUC **99.95%** | Strong in-domain full-audio result.[5] |
| **Speed-invariant log-frequency detector** | Suno v5, speed attack | AUC **99.7%**, F1 **98.6%** | Robust to speed scaling by design; broader corruption and generator coverage remain open.[8] |

MusicDET itself exposes an important limitation: its one-class EER rose from 4.51% to 44.73% under pitch shifting, 44.11% with white noise, and 41.75% after 64 kbps MP3 in its reported robustness study.[7] The newer speed-invariant detector closes one attack class but explicitly does not solve codec, equalization, dynamics, or broad generator generalization.[8]


In [ ]:
#@title 17. Save a reusable model bundle, threshold policy, and model card
bundle = {
    'config': asdict(CFG),
    'config_hash': CONFIG_HASH,
    'feature_columns': FEATURE_COLS,
    'lr_model': lr_model,
    'hgb_model': hgb_model,
    'isotonic_calibrator': iso,
    'real_scaler': real_scaler,
    'real_pca': real_pca,
    'real_covariance': real_cov,
    'real_validation_distances': real_val_dist,
    'thresholds': THRESHOLDS,
}
joblib.dump(bundle, OUTPUT_DIR / 'haim_guard_model.joblib')
with open(OUTPUT_DIR / 'threshold_policy.json', 'w') as handle:
    json.dump(THRESHOLDS, handle, indent=2)
with open(OUTPUT_DIR / 'config.json', 'w') as handle:
    json.dump(asdict(CFG), handle, indent=2, default=list)

worst_corruption = robust_metrics.iloc[0]
model_card = f"""# HAIM-Guard Model Card

## Intended use
Research and human-in-the-loop triage of possible AI involvement in music. Not proof of authorship, copyright infringement, impersonation, or consent.

## Data
Official HAIM public preview audio, CC BY-NC 4.0. Training generators: {CFG.train_generators}. Held-out generator: {CFG.unseen_generator}. Tracks are split before cropping.

## Model
Soft-voting supervised ensemble (logistic regression plus histogram gradient boosting), isotonic calibration, and a real-only PCA plus Ledoit–Wolf Mahalanobis anomaly branch. Feature vector includes log-mel, MFCC, chroma, low-level spectral statistics, and log-frequency shift-insensitive fingerprint features.

## Decision policy
Binary threshold: {binary_threshold:.4f}. Likely-human below {review_low:.4f}; AI indicators at or above {review_high:.4f}; intermediate scores require review. Target validation false-positive rate: {CFG.target_validation_fpr:.1%}.

## Measured performance

{clean_metrics.to_markdown(index=False)}

Worst measured corruption: **{worst_corruption['evaluation']}**, F1={worst_corruption['f1']:.3f}, EER={worst_corruption['eer']:.3f}.

## Known limitations
The quick dataset is small; commercial and future generators may not match the sampled distributions; hybrid role attribution is not solved by a binary score; codecs and signal processing can remove or move forensic cues; dataset collection artifacts can inflate performance; calibration may drift; and adaptive adversaries are not fully modeled.

## Governance
Use human review, preserve original files and hashes, log model/config versions, monitor false positives by genre/language/source, provide appeals, and retrain only with documented licensed data.
"""
(OUTPUT_DIR / 'model_card.md').write_text(model_card)

print('Saved artifacts:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f' - {path.name}: {path.stat().st_size/1024:.1f} KiB')


## Operational recommendations

A robust deployment should combine passive detection with **signed provenance**, account and upload-abuse controls, rate limits, duplicate/fingerprint detection, and human escalation. Keep the original upload and its cryptographic hash, but minimize retention of user data. Monitor drift by generator, genre, language, codec, geography, and upload channel. Recalibrate on a time-based holdout rather than silently changing the decision threshold.

The appropriate automated action is usually **label or review**, not removal or punishment. Human-created experimental music, heavily mastered tracks, low-bitrate archives, and unusual genres can resemble out-of-distribution audio. Conversely, high-quality generators and hybrid workflows can appear human. A grey-zone policy is therefore a security control, not an admission of failure.

### Suggested extensions

For a larger research project, replace the compact anomaly branch with the official [MusicDET](https://github.com/Chaolei98/MusicDET) normalizing-flow implementation, add the public [CLAM](https://github.com/StarkVision-AI/MoM-CLAM) dual-stream model, and evaluate the public [Fusion Segment Transformer](https://github.com/Mippia/FST-AI-music-detection). Train on raw HAIM baseline folders, reserve entire generator families and time periods, and use B/C categories only for final challenge testing.

## References

[1]: https://arxiv.org/abs/2606.01686 "HAIM: Human-AI Music Datasets for AI Music Production Tracking Benchmark"
[2]: https://huggingface.co/datasets/mippia/HAIM "HAIM dataset card and downloads"
[3]: https://github.com/awsaf49/sonics "SONICS dataset, SpecTTTra code, checkpoints, and benchmark"
[4]: https://arxiv.org/abs/2512.00621 "Melody or Machine: Detecting Synthetic Music with Dual-Stream Contrastive Learning"
[5]: https://arxiv.org/abs/2601.13647 "Fusion Segment Transformer for AI-Generated Music Detection"
[6]: https://zenodo.org/records/15063698 "FakeMusicCaps dataset"
[7]: https://arxiv.org/abs/2605.18072 "MusicDET: Zero-Shot AI-Generated Music Detection"
[8]: https://arxiv.org/abs/2607.27454 "Improved Robustness in AI-Generated Music Detection"
[9]: https://singfake.org/ "SingFake: Singing Voice Deepfake Detection"
